In [1]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.graph_objects as go
import base64
import os

# ==============================================================================
# 🎯 參數調整區一：9大館舍廁所資料庫（含總分、星等、總評摘要）
# 說明：後續若要修改分數、星星、評語，直接在對應的雙引號或括號內修改即可。
# 提示：X、Y 座標範圍為 0 到 10，您可以根據執行後點點在您的 back.png 地圖上的位置手動微調。
# ==============================================================================
toilets_data = {
    "國璽樓": {
        "scores": [4.17, 3.50, 4.67, 3.17], "x": 8.5, "y": 7.0,
        "stars": "⭐⭐⭐⭐", "total": "72 / 95",
        "review": "整體乾淨有定期打掃，但注意網路卡不宜久留。"
    },
    "利瑪竇大樓": {
        "scores": [3.67, 3.75, 4.67, 2.8], "x": 6.0, "y": 3.5,
        "stars": "⭐⭐⭐⭐", "total": "68 / 95",
        "review": "雖然缺少部分設施，但不只乾淨也種了綠植，整體偏粉粉的可愛風格。"
    },
    "舊醫學大樓": {
        "scores": [3.00, 3.00, 4.00, 2.17], "x": 6.5, "y": 7.0,
        "stars": "⭐⭐⭐", "total": "55 / 95",
        "review": "沒全身鏡、無障礙設施和性別友善廁所，不只是網路卡，甚至一個人會感到恐怖。"
    },
    "心園": {
        "scores": [2.00, 2.00, 3.67, 1.67], "x": 3.0, "y": 7.5,
        "stars": "⭐⭐⭐", "total": "41 / 95",
        "review": "又悶又暗網路還卡，不單單只是有異味，整體而言又髒又恐怖。"
    },
    "樹德樓": {
        "scores": [4.83, 4.25, 5.00, 3.50], "x": 4.5, "y": 2.5,
        "stars": "⭐⭐⭐⭐⭐", "total": "82 / 95",
        "review": "有防觸電安全設計，定期打掃很乾淨，簡直是高貴奢華風。"
    },
    "伯達樓": {
        "scores": [5.00, 4.25, 5.00, 3.50], "x": 5.5, "y": 3.0,
        "stars": "⭐⭐⭐⭐⭐", "total": "83 / 95",
        "review": "足夠乾淨又寬敞，簡直是有錢人家的廁所。"
    },
    "羅耀拉大樓": {
        "scores": [4.17, 3.50, 5.00, 3.50], "x": 4.0, "y": 4.0,
        "stars": "⭐⭐⭐⭐", "total": "75 / 95",
        "review": "乾淨且設施供應充足，空間寬敞，偏典雅風。"
    },
    "濟時樓": {
        "scores": [4.83, 5.00, 5.00, 3.67], "x": 2.5, "y": 5.0,
        "stars": "⭐⭐⭐⭐⭐", "total": "86 / 95",
        "review": "有衛生紙又定期打掃，寬敞的典雅風，是適合大便的冷氣最強的廁所。"
    },
    "進修部大樓": {
        "scores": [3.83, 4.25, 4.67, 2.50], "x": 5.0, "y": 7.5,
        "stars": "⭐⭐⭐⭐", "total": "69 / 95",
        "review": "些許髒污、網路偏卡，風格介於可愛與髒亂之間。"
    }
}

# ==============================================================================
# 🎯 參數調整區二：背景圖片設定
# ==============================================================================
image_filename = 'back.png'
encoded_image = ""
if os.path.exists(image_filename):
    with open(image_filename, 'rb') as f:
        encoded_image = base64.b64encode(f.read()).decode('utf-8')

# 初始化 App
app = dash.Dash(__name__)

# 版面配置及尺寸設定 (優化右側版面，預留文字區)
MAP_WIDTH = 650
MAP_HEIGHT = 600
RADAR_WIDTH = 500
RADAR_HEIGHT = 440

app.layout = html.Div(style={'backgroundColor': '#1E1E1E', 'padding': '20px', 'display': 'flex', 'fontFamily': 'Microsoft JhengHei'}, children=[
    # 左側：乾淨地圖區
    html.Div(style={'width': '55%'}, children=[
        dcc.Graph(id='fujen-map-back', config={'displayModeBar': False})
    ]),
    # 右側：雷達圖 + 下方動態總評文字區
    html.Div(style={'width': '45%', 'display': 'flex', 'flexDirection': 'column', 'alignItems': 'center'}, children=[
        dcc.Graph(id='radar-chart-back', config={'displayModeBar': False}),
        # 🛠️ 這是動態文字方塊，滑鼠移到哪棟樓，這裡就會自動刷新文字
        html.Div(id='review-box-back', style={
            'width': '90%', 'marginTop': '10px', 'padding': '15px',
            'backgroundColor': 'rgba(255, 255, 255, 0.05)', 'borderRadius': '8px',
            'border': '1px solid rgba(0, 255, 204, 0.3)', 'color': '#FFFFFF'
        })
    ])
])

# 函數 A：繪製左側完全乾淨的校園地圖 (將所有館舍一次點上去)
def draw_base_map():
    fig = go.Figure()
    for name, info in toilets_data.items():
        fig.add_trace(go.Scatter(
            x=[info["x"]], y=[info["y"]],
            mode="markers+text",
            marker=dict(size=18, color='#00FFCC', line=dict(color='#FFFFFF', width=1.5)),
            text=[name],
            textposition="top center",
            textfont=dict(size=13, color="#FFFFFF", family="Microsoft JhengHei"),
            # 🛠️ 關鍵修復：將 X, Y 座標綁進點中，讓後台透過座標100%精準識別是哪棟樓
            customdata=[[info["x"], info["y"]]],
            hoverinfo="text",
            hovertext=f"<b>🏢 {name}</b><br>🔍 游標移入，下方即刻呈現雷達圖與總評",
            hovertemplate="%{hovertext}<extra></extra>",
            showlegend=False
        ))

    img_source = f"data:image/png;base64,{encoded_image}" if encoded_image else ""

    fig.update_layout(
        images=[dict(
            source=img_source, xref="x", yref="y", x=0, y=10, sizex=10, sizey=10,
            sizing="stretch", opacity=0.9, layer="below"
        )],
        title=dict(text="<b>🗺️ 輔大後側區域 — 廁所空間生態地圖</b>", x=0.5, font=dict(size=20, color='#FFFFFF')),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-0.5, 10.5], fixedrange=True),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-0.5, 10.5], fixedrange=True),
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        width=MAP_WIDTH, height=MAP_HEIGHT, margin=dict(l=20, r=20, t=60, b=20)
    )
    return fig

# 函數 B：幾何四角雷達圖的繪製邏輯
def draw_radar(building_name):
    scores = toilets_data[building_name]["scores"]
    s1, s2, s3, s4 = scores

    x_coords = [0, s2, 0, -s4, 0]
    y_coords = [s1, 0, -s3, 0, s1]

    hover_texts = [
        f"1.環境衛生: {s1:.2f}分", f"2.設施完備: {s2:.2f}分",
        f"3.空間舒適: {s3:.2f}分", f"4.特殊機能: {s4:.2f}分", f"1.環境衛生: {s1:.2f}分"
    ]

    fig = go.Figure()

    # 繪製背景「蜘蛛網」正方形直線網格 (1-5分)
    for r in range(1, 6):
        fig.add_trace(go.Scatter(
            x=[0, r, 0, -r, 0], y=[r, 0, -r, 0, r],
            mode='lines', line=dict(color='rgba(255,255,255,0.18)', width=1),
            showlegend=False, hoverinfo='skip'
        ))
        fig.add_trace(go.Scatter(
            x=[0.15], y=[r-0.15], mode='text', text=[str(r)],
            textfont=dict(color='rgba(255,255,255,0.4)', size=10),
            showlegend=False, hoverinfo='skip'
        ))

    # 十字骨架線
    fig.add_trace(go.Scatter(
        x=[-5, 5, None, 0, 0], y=[0, 0, None, -5, 5],
        mode='lines', line=dict(color='rgba(255,255,255,0.2)', width=1.5),
        showlegend=False, hoverinfo='skip'
    ))

    # 繪製中央數據鑽石盾牌
    fig.add_trace(go.Scatter(
        x=x_coords, y=y_coords,
        fill='toself', fillcolor='rgba(0, 200, 150, 0.25)',
        line=dict(color='#00FFCC', width=4), marker=dict(color='#00FFCC', size=9),
        hoverinfo='text', text=hover_texts, hovertemplate="%{text}<extra></extra>"
    ))

    # 固定四角指標文字
    labels = [
        dict(x=0, y=5.5, text="1.環境衛生"), dict(x=5.9, y=0, text="2.設施完備"),
        dict(x=0, y=-5.5, text="3.空間舒適"), dict(x=-5.9, y=0, text="4.特殊機能")
    ]
    for label in labels:
        fig.add_trace(go.Scatter(
            x=[label['x']], y=[label['y']], mode='text', text=[f"<b>{label['text']}</b>"],
            textfont=dict(size=14, color='#00FFCC', family='Microsoft JhengHei'),
            showlegend=False, hoverinfo='skip'
        ))

    fig.update_layout(
        title=dict(text=f"<b>{building_name} 綜合評鑑鑽石圖</b>", x=0.5, font=dict(size=18, color='#FFFFFF')),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-6.8, 6.8], fixedrange=True),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-6.8, 6.8], fixedrange=True),
        showlegend=False, paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        width=RADAR_WIDTH, height=RADAR_HEIGHT, margin=dict(l=20, r=20, t=50, b=10)
    )
    return fig

# 🛠️ 8. 雙重輸出監聽：滑鼠移入點點，同時刷新「右側雷達圖」與「下方文字總評區」
@app.callback(
    [Output('radar-chart-back', 'figure'),
     Output('review-box-back', 'children')],
    Input('fujen-map-back', 'hoverData')
)
def update_dashboard_on_hover(hoverData):
    # 預設初始大樓
    building_name = "國璽樓"

    # 🛠️ 核心修復：直接抓取滑鼠懸停點的精確 X, Y 座標進行比對，100% 繞過緩存與字串 bug
    if hoverData and 'points' in hoverData and len(hoverData['points']) > 0:
        hover_x = hoverData['points'][0].get('x', None)
        hover_y = hoverData['points'][0].get('y', None)

        for b_name, b_info in toilets_data.items():
            if b_info["x"] == hover_x and b_info["y"] == hover_y:
                building_name = b_name
                break

    # 抓取該大樓的總評文字資料
    info = toilets_data[building_name]

    # 建立精美的 HTML 下方文字佈局
    review_layout = html.Div([
        html.Div([
            html.Span(f"🏢 {building_name} ", style={'fontSize': '18px', 'fontWeight': 'bold', 'color': '#00FFCC'}),
            html.Span(f" {info['stars']}", style={'fontSize': '16px', 'marginLeft': '10px'})
        ], style={'display': 'flex', 'alignItems': 'center', 'marginBottom': '8px'}),
        html.Div([
            html.B("📊 綜合實測總分：", style={'color': '#FF9900'}),
            html.Span(info['total'], style={'fontSize': '16px', 'fontWeight': 'bold'})
        ], style={'marginBottom': '8px'}),
        html.Div([
            html.B("📝 實地考察總評：", style={'color': '#00FFFF'}),
            html.P(info['review'], style={'fontSize': '13px', 'color': '#DDDDDD', 'lineHeight': '1.5', 'margin': '0'})
        ])
    ])

    return draw_radar(building_name), review_layout

@app.callback(Output('fujen-map-back', 'figure'), Input('fujen-map-back', 'id'))
def init_map(_): return draw_base_map()
import plotly.io as pio
pio.write_html(draw_base_map(), file="bmap.html", auto_open=False)
if __name__ == '__main__':
    import notebook
    # 🛠️ 只有這兩行！直接利用自動索引去讀取你 toilets_data 的第一間大樓：
    import plotly.io as pio
    pio.write_html(draw_radar(list(toilets_data.keys())[0]), file="back.html", include_plotlyjs="cdn")
    app.run(jupyter_mode='inline', port=8050)

In [2]:
from pyngrok import ngrok

# 🛠️ 直奔主題，直接開啟通道對接 8060
try:
    public_url = ngrok.connect(8050)
    print("\n🎉🎉🎉 終於成功拿到外部互動連結了！！！ 🎉🎉🎉")
    print(f"請複製這串網址發給組員或貼進 Canva：\n\n{public_url}\n")
    print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print("⚠️ 提醒：報告完畢前，請維持地圖專案執行，網頁才能正常互動喔！")
except Exception as e:
    print(f"發生錯誤：{e}")


🎉🎉🎉 終於成功拿到外部互動連結了！！！ 🎉🎉🎉
請複製這串網址發給組員或貼進 Canva：

NgrokTunnel: "https://cofounder-uprising-spiritism.ngrok-free.dev" -> "http://localhost:8050"

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
⚠️ 提醒：報告完畢前，請維持地圖專案執行，網頁才能正常互動喔！
